# AlphaNet Cryptocurrency Data Preparation

This notebook prepares cryptocurrency 15-minute OHLCV data for AlphaNet training, adapting the original stock methodology to crypto markets.

## Data Overview
- **Data Source**: 334 cryptocurrency parquet files with 15-minute intervals
- **Time Range**: ~2021-2024
- **AlphaNet Format**: 9×30 feature matrices with standardized return targets
- **Key Challenge**: Fill time gaps and adapt stock methodology to 24/7 crypto markets

## 1. Environment Setup and Dependencies

In [2]:
import pandas as pd
import numpy as np
import os
import warnings
from datetime import datetime, timedelta
from typing import Dict, List, Tuple, Optional
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed
import psutil
import time
from joblib import Parallel, delayed

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

In [3]:
stable_list = ['USDCUSDT', 'USUALUSDT', 'USUALUSDT', 'TUSDT']

## 2. Robust Data Loading Functions

In [4]:
def read_parquet_robust(file_path: str) -> pd.DataFrame:
    
    """
    Robust parquet file reader with multiple engine fallbacks
    
    Args:
        file_path (str): Path to the parquet file
        
    Returns:
        pd.DataFrame: Loaded dataframe or empty dataframe if failed
    """
    df = pd.read_parquet(file_path)
    if not df.empty:
        # Column mapping for actual parquet structure
        column_mapping = {
            'open_price': 'open',
            'high_price': 'high', 
            'low_price': 'low',
            'close_price': 'close',
            # timestamp and volume are already correctly named
        }

        # Apply column mapping
        df = df.rename(columns=column_mapping)

        # Required columns after mapping
        required_cols = ['timestamp', 'open', 'high', 'low', 'close', 'volume']

        # Convert timestamp to datetime
        if df['timestamp'].dtype == 'object':
            df['timestamp'] = pd.to_datetime(df['timestamp'])
        elif df['timestamp'].dtype in ['int64', 'int32']:
            # Handle millisecond timestamps
            df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')

        # Convert price columns to float
        price_cols = ['open', 'high', 'low', 'close']
        for col in price_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')

        # Convert volume columns to numeric (handle object types)
        volume_cols = ['volume', 'amount', 'buy_volume', 'buy_amount']
        for col in volume_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')

        # Convert count to int
        if 'count' in df.columns:
            df['count'] = pd.to_numeric(df['count'], errors='coerce').fillna(0).astype('int64')

        # Sort by timestamp
        df = df.sort_values('timestamp').reset_index(drop=True)
        df['vwap'] = (df['amount'] / df['volume']).replace([np.inf, -np.inf], np.nan).ffill()

        # Keep only required columns (but also preserve additional useful columns)
        # Keep buy_volume and buy_amount for potential enhanced feature engineering
        available_extra_cols = [col for col in ['amount', 'count', 'buy_volume', 'buy_amount', 'vwap'] if col in df.columns]
        final_cols = required_cols + available_extra_cols
        df = df[final_cols]
    return df


def load_crypto_data(data_dir: str = '../data/raw') -> Dict[str, pd.DataFrame]:
    """
    Load all cryptocurrency parquet files with robust error handling
    
    Returns:
        Dict[str, pd.DataFrame]: Dictionary mapping symbol to dataframe
    """
    print(f" Loading cryptocurrency data from {data_dir}...")
    
    if not os.path.exists(data_dir):
        print(f"Data directory not found: {data_dir}")
        return {}
    
    STABLE = [x + '.parquet' for x in stable_list]
    
    parquet_files = [f for f in os.listdir(data_dir) if f.endswith('.parquet') and f not in STABLE]
    
    parquet_files
    print(f"Found {len(parquet_files)} parquet files")
    
    crypto_data = {}
    successful_loads = 0
    failed_loads = 0
    failed_symbols = []
    
    for filename in tqdm(parquet_files, desc="Loading files"):
        symbol = filename.replace('.parquet', '')
        file_path = os.path.join(data_dir, filename)
        
        try:
            # Load and standardize data
            df = read_parquet_robust(file_path)
            df['symbol'] = symbol
            
            if not df.empty and len(df) >= 100:  # Minimum data requirement
                crypto_data[symbol] = df
                successful_loads += 1
            else:
                failed_loads += 1
                failed_symbols.append(symbol)
                if len(df) < 100:
                    print(f"{symbol}: Insufficient data ({len(df)} records)")
                    
        except Exception as e:
            failed_loads += 1
            failed_symbols.append(symbol)
            print(f"Failed to load {symbol}: {str(e)[:50]}...")
    
    print(f"\\n Successfully loaded: {successful_loads} files")
    print(f"Failed to load: {failed_loads} files")
    
    if failed_symbols:
        print(f"\\n Failed symbols: {', '.join(failed_symbols[:10])}{'...' if len(failed_symbols) > 10 else ''}")
        print(f"Recommendation: These {failed_loads} coins will be excluded from AlphaNet training")
        print(f"This is normal - represents {failed_loads/len(parquet_files)*100:.1f}% failure rate")
    
    return crypto_data

## 2. Load and Process Data

In [5]:
# Load all cryptocurrency data
crypto_data = load_crypto_data()

print(f"\nLoaded data for {len(crypto_data)} cryptocurrencies")

# Show sample data
if crypto_data:
    sample_symbol = list(crypto_data.keys())[0] 
    sample_df = crypto_data[sample_symbol]
    print(f"\nSample data from {sample_symbol}:")
    print(sample_df.head())
    print(f"\nData types:")
    print(sample_df.dtypes)

 Loading cryptocurrency data from ../data/raw...
Found 352 parquet files


Loading files:   7%|████▋                                                             | 25/352 [00:10<01:34,  3.47it/s]

AI16ZUSDT: Insufficient data (0 records)


Loading files:   8%|█████▍                                                            | 29/352 [00:10<00:49,  6.50it/s]

ALCHUSDT: Insufficient data (0 records)


...

VINEUSDT: Insufficient data (0 records)
VTHOUSDT: Insufficient data (0 records)
VVVUSDT: Insufficient data (0 records)


Loading files:  99%|████████████████████████████████████████████████████████████████ | 347/352 [02:13<00:02,  2.31it/s]

ZEREBROUSDT: Insufficient data (0 records)


Loading files: 100%|█████████████████████████████████████████████████████████████████| 352/352 [02:15<00:00,  2.60it/s]

\n Successfully loaded: 331 files
Failed to load: 21 files
\n Failed symbols: AI16ZUSDT, ALCHUSDT, ANIMEUSDT, ARCUSDT, AVAAIUSDT, BIOUSDT, COOKIEUSDT, DUSDT, GRIFFAINUSDT, MELANIAUSDT...
Recommendation: These 21 coins will be excluded from AlphaNet training
This is normal - represents 6.0% failure rate

Loaded data for 331 cryptocurrencies

Sample data from 1000000MOGUSDT:
            timestamp    open    high     low   close     volume  \
0 2024-11-07 12:30:00  2.0896  2.1361  2.0560  2.0652   876381.4   
1 2024-11-07 12:45:00  2.0659  2.0730  1.9797  1.9898  1108330.3   
2 2024-11-07 13:00:00  1.9915  2.0132  1.9521  2.0072  1208569.3   
3 2024-11-07 13:15:00  2.0073  2.0283  1.9631  1.9805  1287539.7   
4 2024-11-07 13:30:00  1.9795  1.9958  1.9158  1.9917  1581100.1   

         amount  count  buy_volume    buy_amount      vwap          symbol  
0  1.824179e+06   6864    414306.5  8.635601e+05  2.081490  1000000MOGUSDT  
1  2.232821e+06   8529    461204.8  9.287305e+05  2.014581  1

In [6]:
abnormal_ones = []
for k in crypto_data:
    temp = crypto_data[k]
    if temp[temp['count'] == 0].shape[0] > 10:
        print(k)
        abnormal_ones.append(k)
        
for k in (abnormal_ones):
    crypto_data.pop(k)

ICPUSDT
TLMUSDT


In [7]:
test_data = pd.concat([crypto_data[k] for k in crypto_data][:60])

In [50]:
def daily_level(df):
    df['open_day'] = df['open'].shift(96)
    df['high_day'] = df['high'].rolling(96).max()
    df['low_day'] = df['low'].rolling(96).min()
    df['close_day'] = df['low'].rolling(96).min()
    df['volume_day'] = df['volume'].rolling(96).sum()
    df['amount_day'] = df['amount'].rolling(96).sum()
    df['count_day'] = df['count'].rolling(96).sum()
    df['vwap_day'] = df['amount_day'] / df['volume_day']
    df['buy_volume_day'] = df['buy_volume'].rolling(96).sum()
    df['buy_amount_day'] = df['buy_amount'].rolling(96).sum()
    return df
    
    
def calc_factor(df: pd.DataFrame, n: int=20):
    """
    一次性计算 8 大方向的单因子（示例版本）
    参数
    ----
    df : pd.DataFrame
        必须包含全部 12 个字段，索引按 timestamp 升序
    n  : int
        回看窗口，默认为 20 根 bar
    """
    df = df.sort_values('timestamp')
    for hz in ['', '_day']:
        # 1) 价格动量类：
        df['momentum' + hz] = (df['close' + hz] / df['close' + hz].shift(n) - 1)

        # 2) 成交量动量类：n 日均量相对最新量比
        df['vma' + hz] = df['volume' + hz] / df['volume' + hz].rolling(n).mean()

        # 3) 价量背离类：OBV 的单期变化
        df['obv_chg' + hz] = np.sign(df['close' + hz] - df['close' + hz].shift()) * df['volume' + hz]

        # 4) 买卖失衡类：成交量买单占比
        df['bsi' + hz] = (df['buy_volume' + hz] - (df['volume' + hz] - df['buy_volume' + hz])) / df['volume' + hz]

        # 5) VWAP 偏离
        df['vwap_dev' + hz] = (df['close' + hz] - df['vwap' + hz]) / df['vwap' + hz]

        # 6) 波动率类：Garman-Klass 波动率（n 日）
        O = df['open' + hz].values
        H = df['high' + hz].values
        L = df['low' + hz].values
        C = df['close' + hz].values

        # 计算对数价格比
        log_hl = np.log(H / L)
        log_co = np.log(C / O)

        # 计算每日波动率贡献
        daily_var = 0.5 * (log_hl)**2 - (2 * np.log(2) - 1) * (log_co)**2

        # 避免负方差（理论上不应该出现）
        daily_var = np.maximum(daily_var, 0)

        # 计算平均日波动率
        df['gk_volatility' + hz] = np.sqrt(np.nanmean(daily_var))

        # 7) 盘口集中度：平均单笔成交数
        df['avg_trade_size' + hz] = df['volume' + hz] / df['count' + hz]

        # 8) 成交额相关：成交额 / 成交额 n 日均值
        df['amt_ratio' + hz] = df['amount' + hz] / df['amount' + hz].rolling(n).mean()

        # 9) MACD信号：指数移动平均线差值，捕捉趋势变化
        exp1 = df['close' + hz].ewm(span=12, adjust=False).mean()  # 12期EMA
        exp2 = df['close' + hz].ewm(span=26, adjust=False).mean()  # 26期EMA
        macd = exp1 - exp2
        df['macd_signal' + hz] = (macd - macd.ewm(span=9, adjust=False).mean()) / df['close' + hz]

        # 10) 布林带位置：价格在布林带中的相对位置，-1到1之间
        bb_mean = df['close' + hz].rolling(20).mean()
        bb_std = df['close' + hz].rolling(20).std()
        df['bb_position' + hz] = (df['close' + hz] - bb_mean) / (2 * bb_std)

        # 11) 威廉指标：衡量超买超卖，-100到0之间
        highest_high = df['high' + hz].rolling(14).max()
        lowest_low = df['low' + hz].rolling(14).min()
        df['williams_r' + hz] = -100 * (highest_high - df['close' + hz]) / (highest_high - lowest_low)

        # 12) 随机指标K值：价格在最近高低区间的相对位置，0-100
        df['stoch_k' + hz] = 100 * (df['close' + hz] - lowest_low) / (highest_high - lowest_low)

        typical_price = (df['high' + hz] + df['low' + hz] + df['close' + hz]) / 3
        # 14) 资金流量指数（MFI）：结合价格和成交量的动量指标，0-100
        money_flow = typical_price * df['volume' + hz]
        positive_flow = money_flow.where(typical_price > typical_price.shift(), 0)
        negative_flow = money_flow.where(typical_price < typical_price.shift(), 0)
        mf_ratio = positive_flow.rolling(14).sum() / negative_flow.rolling(14).sum()
        df['mfi' + hz] = 100 - (100 / (1 + mf_ratio))

        # 22) 价格效率：收盘价与VWAP的比率，衡量价格发现效率
        df['price_efficiency' + hz] = df['close' + hz] / df['vwap' + hz]

        # 23) K线内波动率：高低价差相对于开盘价的比例
        df['intrabar_vol' + hz] = (df['high' + hz] - df['low' + hz]) / df['open' + hz]
    
    return df


def add_feat(df: pd.DataFrame) -> tuple[pd.DataFrame, list]:
    """优化后的特征工程函数"""
    # 排序数据
    df = df.sort_values(["symbol", "timestamp"]).reset_index(drop=True)
    
    # 计算目标变量 y_raw
    df["future_vwap"] = df.groupby("symbol")["vwap"].shift(-96)
    df["y_raw"] = df["future_vwap"] / df["vwap"] - 1
    
    # 计算特征
    df = df.groupby('symbol', group_keys=False).apply(daily_level)
    df = df.groupby('symbol', group_keys=False).apply(calc_factor)
    
    # 特征列列表
    feature = ['momentum', 'vma', 'obv_chg', 'bsi', 'vwap_dev', 'gk_volatility', 'macd_signal', 'bb_position', 'williams_r',
                    'stoch_k', 'avg_trade_size', 'amt_ratio', 'price_efficiency', 'intrabar_vol', 'mfi']
    feature_cols = feature + [x + '_day' for x in feature]
    
    df = df.dropna()
    # =============== 优化后的去极值处理 ===============
    # 1. 向量化计算中位数和MAD
    grouped = df.groupby('timestamp')
    
    # 一次性计算所有特征的中位数
    medians = grouped[feature_cols].median().add_prefix('median_')
    
    # 一次性计算所有特征的MAD
    abs_devs = df[feature_cols].sub(medians.loc[df['timestamp']].values)
    abs_devs = abs_devs.abs()
    mads = grouped[abs_devs.columns].median().add_prefix('mad_')
    
    # 计算上下界
    k = 3.0
    scale_factor = 1.4826
    upper_bounds = pd.DataFrame(medians.values + (k * scale_factor * mads).values, index=medians.index, columns=feature_cols)
    lower_bounds = pd.DataFrame(medians.values - (k * scale_factor * mads).values, index=medians.index, columns=feature_cols)
    
    # 2. 向量化裁剪
    for col in feature_cols:
        col_vals = df[col].values
        upper = upper_bounds[col].loc[df['timestamp']].values
        lower = lower_bounds[col].loc[df['timestamp']].values
        
        # 使用numpy.clip进行向量化操作
        clipped = np.clip(col_vals, lower, upper)
        df[col] = clipped

    # 2. 标准化处理
    min_ = grouped[feature_cols + ['y_raw']].min()
    max_ = grouped[feature_cols + ['y_raw']].max()

    # 标准化每个特征
    for col in feature_cols + ['y_raw']:
        # 获取当前时间戳的均值和标准差
        col_vals = df[col].values
        min_vals = min_[col].loc[df['timestamp']].values
        max_vals = max_[col].loc[df['timestamp']].values
        
        # 应用标准化公式: (x - mean) / std
        normalized = (col_vals - min_vals) / max_vals - min_vals
        df[col] = normalized
    
    return df.reset_index(drop=True), feature_cols

In [51]:
df_final, features = add_feat(test_data)
df_final.drop(['future_vwap'], axis=1, inplace=True)
# df_final.to_parquet("panel.parquet")

In [52]:
df_final.dropna()

,timestamp,open,high,low,close,volume,amount,count,buy_volume,buy_amount,vwap,symbol,y_raw,open_day,high_day,low_day,close_day,volume_day,amount_day,count_day,vwap_day,buy_volume_day,buy_amount_day,momentum,vma,obv_chg,bsi,vwap_dev,gk_volatility,avg_trade_size,amt_ratio,macd_signal,bb_position,williams_r,stoch_k,mfi,price_efficiency,intrabar_vol,momentum_day,vma_day,obv_chg_day,bsi_day,vwap_dev_day,gk_volatility_day,avg_trade_size_day,amt_ratio_day,macd_signal_day,bb_position_day,williams_r_day,stoch_k_day,mfi_day,price_efficiency_day,intrabar_vol_day
29,2024-11-09 17:00:00,1.94870,1.95510,1.91860,1.92430,213420.2,413055.80798,5170,84832.8,164241.82876,1.935411,1000000MOGUSDT,0.225728,1.96240,2.10000,1.91670,1.91670,15087397.2,3.024406e+07,249641.0,2.004591,7327265.8,1.469439e+07,0.136439,-0.852963,3.780452e+06,1.529879,0.032168,0.502674,-0.855494,-0.850947,0.005878,6.201284,527.139505,0.461765,-22.058721,-0.985380,0.290139,0.003309,-0.846789,4.882821e+06,0.120067,0.224314,0.487516,-0.784597,-0.849674,0.062783,2.916706,538.589318,0.000000,0.000000,-0.879441,0.314185
30,2024-11-09 17:15:00,1.92400,1.94500,1.91000,1.94330,215225.1,413999.89348,3469,86713.7,167124.40382,1.923567,1000000MOGUSDT,0.223668,1.94160,2.10000,1.91000,1.91000,15059313.3,3.018598e+07,249616.0,2.004472,7287618.8,1.461611e+07,0.080656,-0.572921,-6.982009e+03,0.039307,0.460057,0.449343,-0.697796,-0.563072,0.005576,2.116621,294.584741,-30.706175,-16.925557,-0.998126,0.210425,0.007206,-0.625084,1.512731e+08,0.123016,0.256759,0.440849,-0.786275,-0.601876,0.321459,2.703315,531.160500,0.000000,0.000000,-0.858136,0.311011
31,2024-11-09 17:30:00,1.94330,1.94470,1.92990,1.93650,91889.2,177960.06753,1794,44814.2,86817.69150,1.936681,1000000MOGUSDT,0.238355,1.95160,2.10000,1.91000,1.91000,14855045.6,2.978150e+07,247338.0,2.004807,7174429.5,1.439230e+07,0.060447,-0.016577,2.601993e+05,0.142323,0.556715,0.449343,-0.749922,-0.013989,0.004676,2.153414,270.753107,-26.123772,-23.523251,-0.994368,0.350025,0.009233,-0.646589,0.000000e+00,0.131335,0.214554,0.440849,-0.787320,-0.625427,0.622451,1.777703,482.811812,0.000000,0.000000,-0.880557,0.313065
32,2024-11-09 17:45:00,1.93570,1.96070,1.93260,1.95930,147596.4,287315.88688,2205,86641.8,168661.64044,1.946632,1000000MOGUSDT,0.243731,1.96920,2.10000,1.91000,1.91000,14424754.0,2.892193e+07,244101.0,2.005021,6957434.2,1.395829e+07,0.014891,0.443492,5.819435e+04,1.345679,1.188233,0.449343,-0.585652,0.440094,0.002183,0.370672,197.544606,-34.558534,-27.304180,-0.990148,0.597948,0.009233,-0.678910,0.000000e+00,0.142560,0.203734,0.440849,-0.788154,-0.659870,0.414669,1.407165,467.820873,0.000000,0.000000,-0.879665,0.313285
44,2024-11-10 17:15:00,2.14460,2.16980,2.14000,2.15560,83842.9,180298.54151,2158,46792.6,100605.22375,2.150433,1000000MOGUSDT,0.191148,1.92400,2.24980,1.92990,1.92990,23552407.7,4.848778e+07,390337.0,2.058718,11323649.4,2.331993e+07,0.667420,-0.171799,1.181659e+06,1.701461,0.673276,0.449343,-0.447271,-0.175321,1.633539,1.144261,109.477321,-22.043459,-29.659669,-0.987972,0.429701,0.164645,-0.673405,-6.968236e+05,0.146935,0.371082,0.440849,-0.824179,-0.684025,0.567615,-0.755686,502.624405,-1.471215,-100.000000,-0.575297,0.166203
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1772996,2024-12-28 07:30:00,0.14210,0.14227,0.14152,0.14167,805050.0,114202.84959,1535,417048.0,59151.03337,0.141858,BIGTIMEUSDT,0.163645,0.14329,0.15215,0.14133,0.14133,225746000.0,3.301430e+07,264780.0,0.146245,108094172.0,1.579468e+07,0.062287,-0.233051,3.580681e+06,0.632519,0.009350,0.318768,-0.601792,-0.223963,0.002291,3.966558,469.401635,-1.829252,-15.544903,-0.982793,0.034737,0.832862,-0.742538,2.954528e+08,0.128678,0.192270,0.296896,-0.685939,-0.746433,1.230545,3.347925,518.221975,0.294685,1.000000,-0.531678,0.012916
1773026,2024-12-28 16:15:00,0.14281,0.14399,0.14267,0.14351

In [53]:
df_final.dropna(inplace=True)

# 上面是改动后的数据准备，截面标准化只对features列做

# 可以尝试不用原始的数据，只用features输入到模型中预测

In [37]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import torch.optim as optim
import math
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [61]:
class AlphaNetUniversalDataset(Dataset):
    def __init__(self, df, y_columns='y_raw', seq_len=10, min_columns=['momentum', 'vma', 'obv_chg', 'bsi', 'vwap_dev', 'gk_volatility', 'macd_signal', 'bb_position', 'williams_r',
                    'stoch_k', 'avg_trade_size', 'amt_ratio', 'price_efficiency', 'intrabar_vol', 'mfi'],
            daily_columns=[x + '_day' for x in ['momentum', 'vma', 'obv_chg', 'bsi', 'vwap_dev', 'gk_volatility', 'macd_signal', 'bb_position', 'williams_r',
                    'stoch_k', 'avg_trade_size', 'amt_ratio', 'price_efficiency', 'intrabar_vol', 'mfi']]):
        self.seq_len = seq_len
        self.y_columns = y_columns

        # 存储每个symbol的x和y数据
        self.x_list = []  # 存储每个symbol的滑动窗口视图
        self.x_daily_list = []
        self.y_list = []  # 存储每个symbol的y值
        self.info_list = []

        # 存储每个symbol的样本数量
        self.sample_counts = []

        # 存储每个symbol的起始索引
        self.start_indices = []

        # 处理每个symbol的数据
        current_start = 0
        df = df.sort_values(['symbol', 'timestamp']).reset_index(drop=True)
        for symbol, group in df.groupby('symbol'): 
            if group.shape[0] < seq_len * 96:
                # 如果数据长度不足，跳过该symbol
                continue

            # 创建滑动窗口视图
            windowed_data = sliding_window_view(group[min_columns].values, window_shape=seq_len, axis=0).transpose(0, 2, 1)
            windowed_daily_data = sliding_window_view(group[daily_columns].values, window_shape=96 * seq_len, axis=0).transpose(0, 2, 1)
            start = windowed_data.shape[0] - windowed_daily_data.shape[0]
            windowed_data = windowed_data[start:]
            windowed_data = windowed_data[::(seq_len - 1), :, :]
            indices = [i * 96 for i in range(seq_len)]
            selected_view = windowed_daily_data[:, indices, :]
            selected_view = selected_view[::(seq_len - 1), :, :]
            # 存储数据
            self.x_list.append(windowed_data)
            self.y_list.append(group[y_columns].values[start:][::9])
            self.info_list.append(group.index.values[start:][::9])

            self.x_daily_list.append(selected_view)
            # 记录样本数量
            n_samples = windowed_data.shape[0]
            self.sample_counts.append(n_samples)

            # 记录起始索引
            self.start_indices.append(current_start)
            current_start += n_samples

        # 总样本数
        self.total_samples = current_start

        # 创建累积样本数数组，用于快速查找
        self.cumulative_counts = np.cumsum([0] + self.sample_counts)

    def _find_symbol_index(self, idx):
        """根据全局索引找到对应的symbol索引和局部索引"""
        # 使用二分查找找到对应的symbol索引
        symbol_idx = np.searchsorted(self.cumulative_counts, idx, side='right') - 1

        # 计算在该symbol中的局部索引
        local_idx = idx - self.cumulative_counts[symbol_idx]

        return symbol_idx, local_idx

    def __len__(self):
        return self.total_samples

    def __getitem__(self, idx):
        # 找到对应的symbol索引和局部索引
        symbol_idx, local_idx = self._find_symbol_index(idx)

        # 获取对应的数据
        x = self.x_list[symbol_idx][local_idx]
        x_daily = self.x_daily_list[symbol_idx][local_idx]
        x_full = np.concatenate([x, x_daily], axis=-1)
        y = self.y_list[symbol_idx][local_idx]
        info = self.info_list[symbol_idx][local_idx]

        # 转换为PyTorch张量
        x_tensor = torch.from_numpy(x_full.astype(np.float32))  # 1, len_seq: default 96, features 
        y_tensor = torch.tensor(y, dtype=torch.float32) # 1
        info_tensor = torch.tensor(info, dtype=torch.int64)  # 转换为整数张量
        return x_tensor, y_tensor, info_tensor

In [65]:
class CryptoGRU(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.3, bidirectional=False):
        """
        GRU模型用于加密货币收益预测
        
        参数:
        - input_size: 输入特征维度 (n_features)
        - hidden_size: GRU隐藏层大小
        - num_layers: GRU层数
        - dropout: Dropout比例
        - bidirectional: 是否使用双向GRU
        """
        super(CryptoGRU, self).__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1
        
        # GRU层
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=bidirectional
        )
        
        # 注意力机制
        self.attention = nn.Sequential(
            nn.Linear(hidden_size * self.num_directions, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 1)
        )
        
        # 输出层
#         self.fc = nn.Sequential(
#             nn.Linear(hidden_size * self.num_directions, hidden_size // 2),
#             # nn.LayerNorm(hidden_size // 2),
#             nn.ReLU(),
#             nn.Dropout(dropout),
#             nn.Linear(hidden_size // 2, 1),
#             nn.Tanh()
#         )
        self.fc = nn.Sequential(
            nn.Linear(hidden_size * self.num_directions, hidden_size // 2),
            nn.LayerNorm(hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, hidden_size // 4),
            nn.LayerNorm(hidden_size // 4),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 4, 1),
            nn.Sigmoid()
        )
        
        # 初始化权重
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.GRU):
                for name, param in m.named_parameters():
                    if 'weight_ih' in name or 'weight_hh' in name:
                        nn.init.orthogonal_(param)
                    elif 'bias' in name:
                        nn.init.zeros_(param)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    
    def forward(self, x):
        """
        前向传播
        
        参数:
        - x: 输入序列，形状为 [batch_size, seq_len, input_size]
        
        返回:
        - 预测值，形状为 [batch_size]
        """
        batch_size = x.size(0)
        
        # GRU前向传播
        gru_out, _ = self.gru(x)  # [batch_size, seq_len, hidden_size * num_directions]
        
        # 注意力机制
        attn_weights = F.softmax(self.attention(gru_out), dim=1)  # [batch_size, seq_len, 1]
        context = torch.sum(attn_weights * gru_out, dim=1)  # [batch_size, hidden_size * num_directions]
        
        # 输出层
        out = self.fc(context)  # [batch_size, 1]
        
        return out.squeeze(-1)  # [batch_size]
    

class SimpleCryptoGRU(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=1, dropout=0.2):
        """
        简化的GRU模型
        
        参数:
        - input_size: 输入特征维度
        - hidden_size: GRU隐藏层大小
        - num_layers: GRU层数
        - dropout: Dropout比例
        """
        super(SimpleCryptoGRU, self).__init__()
        
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        # GRU前向传播
        gru_out, _ = self.gru(x)  # [batch_size, seq_len, hidden_size]
        
        # 取最后一个时间步的输出
        last_output = gru_out[:, -1, :]  # [batch_size, hidden_size]
        
        # 输出层
        out = self.fc(last_output)  # [batch_size, 1]
        
        return out.squeeze(-1)  # [batch_size]

In [76]:
def train_model(model, train_loader, val_loader, num_epochs=10, learning_rate=0.001, patience=10):
    """
    训练GRU模型
    
    参数:
    - model: GRU模型实例
    - train_loader: 训练数据加载器
    - val_loader: 验证数据加载器
    - num_epochs: 训练轮数
    - learning_rate: 学习率
    - patience: 早停耐心值
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    
    # 定义损失函数和优化器
    criterion = nn.MSELoss()
#     optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-5)
    optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4   # 可以尝试 1e-5 到 1e-3 范围
    )
    
#     scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
#     optimizer, mode='min', factor=0.5, patience=3, verbose=True)

    # 记录训练历史
    train_losses = []
    val_losses = []
    best_val_loss = float('inf')
    patience_counter = 0
    
    for epoch in range(num_epochs):
        # 训练阶段
        model.train()
        train_loss = 0
        step = 0
        for (data, target, info) in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs} - Training"):
            data, target = data.to(device), target.to(device)
            
            # 前向传播
            output = model(data)
            loss = criterion(output, target)
            
            # 反向传播和优化
            optimizer.zero_grad()
            loss.backward()
            
            total_norm = 0
            for name, p in model.named_parameters():
                if p.grad is not None:
                    param_norm = p.grad.detach().data.norm(2)
                    total_norm += param_norm.item() ** 2
            total_norm = total_norm ** 0.5        # 全局 L2 范数
            print(f"step={step}  |  grad_norm={total_norm:.3f}")
            step += 1
            total_norm = 5
    
            
            # 梯度裁剪，防止梯度爆炸
            clip_value = min(10.0, total_norm * 1.2) if total_norm else 1.0
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_value)
            optimizer.step()
            
            train_loss += loss.item()
        
        # 验证阶段
        model.eval()
        val_loss = 0
        all_preds = []
        all_targets = []
        
        with torch.no_grad():
            for (data, target, info) in val_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                
                val_loss += criterion(output, target).item()
                all_preds.extend(output.cpu().numpy())
                all_targets.extend(target.cpu().numpy())
        
        # 计算平均损失
        train_loss /= len(train_loader)
        val_loss /= len(val_loader)
        
        # 计算评估指标
        mse = mean_squared_error(all_targets, all_preds)
        mae = mean_absolute_error(all_targets, all_preds)
        
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        
#         # 学习率调整
#         scheduler.step(val_loss)
        
        # 早停检查
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            # 保存最佳模型
            torch.save(model.state_dict(), 'best_model.pth')
        else:
            patience_counter += 1
        
        # 打印训练进度
        if True:
            print(f'Epoch [{epoch+1}/{num_epochs}], '
                  f'Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}, '
                  f'MSE: {mse:.6f}, MAE: {mae:.6f}')
        
        import gc
        del data, target, output, loss
        gc.collect()
        torch.cuda.empty_cache()

        # 早停
        if patience_counter >= patience:
            print(f'Early stopping at epoch {epoch+1}')
            break
    
    # 加载最佳模型
    model.load_state_dict(torch.load('best_model.pth'))
    
    return model, train_losses, val_losses

In [41]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [42]:
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset

In [77]:
full_dataset = AlphaNetUniversalDataset(df_final, seq_len=20)

train_idx, val_idx = train_test_split(
    list(range(len(full_dataset))),
    train_size=0.6,
    shuffle=False  # 关键：保持顺序，不打乱
)

train_ds = Subset(full_dataset, train_idx)
val_ds = Subset(full_dataset, val_idx)

train_loader = DataLoader(
    train_ds,
    batch_size=100,
    shuffle=True,
)

val_loader = DataLoader(
    val_ds,
    batch_size=100,
    shuffle=False,
)


In [78]:
model = CryptoGRU(input_size=len(features), hidden_size=96, num_layers=1, dropout=0.2)
# model = SimpleCryptoGRU(input_size=len(features))

In [79]:
trained_model, train_losses, val_losses = train_model(
    model, train_loader, val_loader, num_epochs=10, learning_rate=0.001
)

Epoch 1/10 - Training:  46%|███████████████████████████▍                               | 13/28 [00:00<00:00, 61.78it/s]

step=0  |  grad_norm=5.195
step=1  |  grad_norm=90.854
step=2  |  grad_norm=255.054
step=3  |  grad_norm=4.862
step=4  |  grad_norm=5.608
step=5  |  grad_norm=1.442
step=6  |  grad_norm=6.845
step=7  |  grad_norm=374.761
step=8  |  grad_norm=3.185
step=9  |  grad_norm=16.705
step=10  |  grad_norm=12.745
step=11  |  grad_norm=5.200
step=12  |  grad_norm=20.777


Epoch 1/10 - Training: 100%|███████████████████████████████████████████████████████████| 28/28 [00:00<00:00, 68.06it/s]


step=13  |  grad_norm=6.057
step=14  |  grad_norm=7.866
step=15  |  grad_norm=29.977
step=16  |  grad_norm=89.091
step=17  |  grad_norm=2.380
step=18  |  grad_norm=119.923
step=19  |  grad_norm=13.502
step=20  |  grad_norm=1.224
step=21  |  grad_norm=6.698
step=22  |  grad_norm=24.282
step=23  |  grad_norm=3.097
step=24  |  grad_norm=8.261
step=25  |  grad_norm=1.977
step=26  |  grad_norm=8.541
step=27  |  grad_norm=2.578
Epoch [1/10], Train Loss: 257.360100, Val Loss: 68.912575, MSE: 70.414513, MAE: 1.490347


Epoch 2/10 - Training:  29%|█████████████████▏                                          | 8/28 [00:00<00:00, 73.47it/s]

...

Epoch [9/10], Train Loss: 256.604232, Val Loss: 68.888141, MSE: 70.381592, MAE: 1.565691


Epoch 10/10 - Training:  57%|█████████████████████████████████▏                        | 16/28 [00:00<00:00, 76.61it/s]

step=0  |  grad_norm=4.536
step=1  |  grad_norm=2.513
step=2  |  grad_norm=1.440
step=3  |  grad_norm=3.753
step=4  |  grad_norm=1.654
step=5  |  grad_norm=107.344
step=6  |  grad_norm=0.508
step=7  |  grad_norm=0.395
step=8  |  grad_norm=2.016
step=9  |  grad_norm=9.429
step=10  |  grad_norm=31.231
step=11  |  grad_norm=0.391
step=12  |  grad_norm=1.520
step=13  |  grad_norm=1.826
step=14  |  grad_norm=1.340
step=15  |  grad_norm=0.766


Epoch 10/10 - Training: 100%|██████████████████████████████████████████████████████████| 28/28 [00:00<00:00, 77.72it/s]


step=16  |  grad_norm=1.452
step=17  |  grad_norm=2.303
step=18  |  grad_norm=1.881
step=19  |  grad_norm=2.949
step=20  |  grad_norm=4.507
step=21  |  grad_norm=1.093
step=22  |  grad_norm=0.475
step=23  |  grad_norm=0.413
step=24  |  grad_norm=0.418
step=25  |  grad_norm=1.469
step=26  |  grad_norm=0.238
step=27  |  grad_norm=8.219
Epoch [10/10], Train Loss: 260.611356, Val Loss: 68.882558, MSE: 70.375679, MAE: 1.570330


In [133]:
def evaluate_icir(predictions):
    """
    计算ICIR评估指标
    :param predictions: DataFrame包含列 ['date', 'pred', 'true']
    :return: 包含各项指标的字典
    """
    # 按日期分组计算每日IC
    daily_ic = []
    dates = sorted(predictions['date'].unique())

    for date in dates:
        date_data = predictions[predictions['date'] == date]

        # 计算秩相关系数 (Spearman)
        ic = date_data[['pred', 'true']].corr(method='spearman').iloc[0, 1]
        daily_ic.append(ic)

    plot_ic_series(pd.Series(daily_ic, index=dates))

    # 转换为numpy数组
    daily_ic = np.array(daily_ic)

    # 计算指标
    mean_ic = np.mean(daily_ic)
    std_ic = np.std(daily_ic)
    ir = mean_ic / std_ic if std_ic > 0 else 0

    # IC为正的比例
    positive_ic_ratio = np.mean(daily_ic > 0)

    # IC的T检验 (检验IC是否显著大于0)
    from scipy import stats
    t_stat, p_value = stats.ttest_1samp(daily_ic, 0)

    return {
        'daily_ic': daily_ic,
        'mean_ic': mean_ic,
        'std_ic': std_ic,
        'ir': ir,
        'positive_ic_ratio': positive_ic_ratio,
        'ic_t_stat': t_stat,
        'ic_p_value': p_value
    }


def plot_ic_series(ic_values):
    """绘制IC序列图"""
    fig, ax1 = plt.subplots(figsize=(8, 4))

    # 左轴：IC
    ax1.plot(ic_values.index, ic_values,
             color='tab:blue', label='IC', lw=0.5)
    ax1.set_ylabel('IC', color='blue')
    ax1.tick_params(axis='y', labelcolor='blue')

    # 右轴：累计 IC
    ax2 = ax1.twinx()
    ax2.plot(ic_values.index, ic_values.cumsum(),
             color='red', label='Cum IC', lw=0.5)
    ax2.set_ylabel('Cumulative IC', color='red')
    ax2.tick_params(axis='y', labelcolor='red')

    # 图例：把两个轴的 label 放到一起
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

    plt.title('IC vs Cumulative IC')
    plt.tight_layout()
    plt.show()